# 01. GCR Baseline Reproduction

**Purpose:** Reproduce the Graph-Constrained Reasoning (GCR) baseline on WebQSP and CWQ.

**Output:** Saved predictions JSONL at `results/GenPaths/{dataset}/GCR-{model}/test/{config}/predictions.jsonl`

**Reference:** Luo et al. (2025) — GCR achieves 92.6 Hits@1 on WebQSP, 75.8 on CWQ.

Based on: `workflow/predict_paths_and_answers.py` + `scripts/graph_constrained_decoding.sh`

**Note:** This is the unmodified GCR baseline, used to establish the reference SIR and Hits@1 for comparison with DCA-Trie variants in notebooks 02-04.

In [ ]:
# Setup & Environment
import os
import sys
import time
import json
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="transformers")

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive', force_remount=False)
    DRIVE_BASE = "/content/drive/MyDrive/GCR-Results"
    os.makedirs(DRIVE_BASE, exist_ok=True)
except ImportError:
    IN_COLAB = False
    DRIVE_BASE = None
    print("Not on Colab — skipping Drive mount.")

# GPU Detection
import torch

GPU_ARCH_MAP = {
    "A100": "80",
    "A100-SXM": "80",
    "A100-PCIE": "80",
    "L4": "89",
    "T4": "75",
    "V100": "70",
    "H100": "90",
    "H200": "90",
    "RTX 4090": "89",
    "RTX 3090": "86",
}

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_arch = None
    for key, arch in GPU_ARCH_MAP.items():
        if key in gpu_name:
            gpu_arch = arch
            break
    has_a100 = "A100" in gpu_name
    print(f"GPU: {gpu_name}  |  VRAM: {gpu_mem:.1f} GB  |  SM arch: {gpu_arch}")
else:
    gpu_name = "None"
    gpu_mem = 0
    gpu_arch = None
    has_a100 = False
    print("WARNING: No GPU detected. Inference will be very slow.")

# Clone repo
REPO_DIR = "/content/graph-constrained-reasoning" if IN_COLAB else os.getcwd()
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Adjanour/graph-constrained-reasoning-dca {REPO_DIR}

if IN_COLAB:
    os.chdir(REPO_DIR)
sys.path.insert(0, '.')

# Pinned dependencies
DEPS = [
    "transformers==4.44.2",
    "accelerate>=0.30.1",
    "datasets>=2.19.2",
    "marisa-trie>=1.2.0",
    "networkx>=3.0",
    "scikit-learn>=1.5.0",
    "tiktoken>=0.7.0",
    "sentencepiece>=0.2.0",
    "protobuf>=5.27.1",
]

print("Installing core dependencies...")
!pip install -q {' '.join(DEPS)}
print("Done.")

# Install flash-attn (optional, ~20% faster on A100)
flash_attn_installed = False

if torch.cuda.is_available():
    py_ver = f"{sys.version_info.major}{sys.version_info.minor}"
    cuda_ver = torch.version.cuda.replace(".", "")
    torch_ver = torch.__version__.split("+")[0]
    torch_major_minor = ".".join(torch_ver.split(".")[:2])

    wheel_url = (
        f"https://github.com/Dao-AILab/flash-attention/releases/"
        f"download/v2.7.3/flash_attn-2.7.3+cu{cuda_ver}"
        f"torch{torch_major_minor}cxx11abiFALSE-cp{py_ver}"
        f"-cp{py_ver}-linux_x86_64.whl"
    )
    print("Trying pre-compiled flash-attn wheel...")
    ret = os.system(f"pip install -q '{wheel_url}' 2>/dev/null")
    if ret == 0:
        flash_attn_installed = True
        print("flash-attn installed from pre-compiled wheel.")
    else:
        print("Pre-compiled wheel not available. Using sdpa.")

if flash_attn_installed:
    print("flash_attention_2 available.")
else:
    print("Using sdpa attention (built into transformers, no install needed).")

# Imports
from tqdm import tqdm
from datasets import load_dataset
from src.llms import get_registed_model
from src.qa_prompt_builder import PathGenerationWithAnswerPromptBuilder
from src.utils.qa_utils import eval_path_result_w_ans
import src.utils as utils
import networkx as nx

print("All imports verified.")

## Configuration

In [ ]:
# Model
MODEL_PATH = "rmanluo/GCR-Meta-Llama-3.1-8B-Instruct"
MODEL_NAME = "rmanluo/GCR-Meta-Llama-3.1-8B-Instruct"

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
except (ImportError, Exception):
    pass

# Datasets
DATA_PATH = "rmanluo"
DATASETS = ["RoG-webqsp", "RoG-cwq"]
SPLIT = "test"

# Decoding
INDEX_LEN = 2
K = 5
GEN_MODE = "group-beam"
PROMPT_MODE = "zero-shot"
MAX_NEW_TOKENS = 256
N_PROC = 1
ATTN_IMPL = "flash_attention_2" if (has_a100 and flash_attn_installed) else "sdpa"

# Sampling
MAX_SAMPLES = 100

# Output
PREDICT_PATH = "results/GenPaths"
FORCE_RERUN = True

print(f"Model: {MODEL_PATH}")
print(f"Attention: {ATTN_IMPL}")
print(f"Datasets: {DATASETS}")
print(f"Beam width k: {K}")
print(f"Max path length: {INDEX_LEN}")
print(f"Max samples: {MAX_SAMPLES or 'full'}")

## Load Model

In [ ]:
import argparse

LLM = get_registed_model(MODEL_NAME)

parser = argparse.ArgumentParser()
LLM.add_args(parser)
args = parser.parse_args([])
args.model_path = MODEL_PATH
args.model_name = MODEL_NAME
args.k = K
args.generation_mode = GEN_MODE
args.attn_implementation = ATTN_IMPL
args.max_new_tokens = MAX_NEW_TOKENS
args.maximun_token = 4096

print("Loading model (this takes 1-2 minutes)...")
try:
    model = LLM(args)
    model.prepare_for_inference()
    model.generation_cfg.temperature = None
    model.generation_cfg.top_p = None
    model.generation_cfg.top_k = None
    model.model.generation_config.temperature = None
    model.model.generation_config.top_p = None
    model.model.generation_config.top_k = None
    print("Model loaded.")
except Exception as e:
    print(f"Failed to load model: {e}")
    raise

## Run GCR Baseline

In [ ]:
def get_output_file(path, force=False):
    """Open predictions file for writing, resuming from existing data if not forcing."""
    if not os.path.exists(path) or force:
        fout = open(path, "w")
        return fout, []
    else:
        with open(path, "r") as f:
            processed = [json.loads(line)["id"] for line in f]
        fout = open(path, "a")
        return fout, processed


def predict_one(data, processed_list, input_builder, model):
    """Run GCR prediction for a single question. Returns result dict or None."""
    qid = data["id"]
    if qid in processed_list:
        return None
    input_query, ground_paths, trie = input_builder.process_input(data)
    if trie is None:
        return None
    start_token_ids = model.tokenizer.convert_tokens_to_ids(input_builder.PATH_START_TOKEN)
    end_token_ids = model.tokenizer.convert_tokens_to_ids(input_builder.PATH_END_TOKEN)
    llm_input = model.prepare_model_prompt(input_query)
    prediction = model.generate_sentence(
        llm_input, trie,
        start_token_ids=start_token_ids,
        end_token_ids=end_token_ids,
        enable_constrained_by_default=False,
    )
    if prediction is None:
        return None
    return {
        "id": qid,
        "question": data["question"],
        "prediction": prediction,
        "ground_truth": data["answer"],
        "ground_truth_paths": ground_paths,
        "input": input_query,
    }


SELECTED_DATASET = "RoG-webqsp"

print(f"Processing {SELECTED_DATASET}...")
dataset = load_dataset(f"{DATA_PATH}/{SELECTED_DATASET}", split=SPLIT)
print(f"Loaded {len(dataset)} questions")

if MAX_SAMPLES is not None:
    dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))
    print(f"Subsampled to {len(dataset)} questions")

input_builder = PathGenerationWithAnswerPromptBuilder(
    model.tokenizer, PROMPT_MODE, index_path_length=INDEX_LEN
)

postfix = f"{PROMPT_MODE}-{GEN_MODE}-k{K}-index_len{INDEX_LEN}"
model_short = MODEL_PATH.split("/")[-1]
output_dir = os.path.join(PREDICT_PATH, SELECTED_DATASET, model_short, SPLIT, postfix)
os.makedirs(output_dir, exist_ok=True)
print(f"Output: {output_dir}")

predictions_file = os.path.join(output_dir, "predictions.jsonl")

try:
    fout, processed = get_output_file(predictions_file, force=FORCE_RERUN)
    print(f"Already processed: {len(processed)} questions")

    for data in tqdm(dataset, desc="Generating"):
        try:
            res = predict_one(data, processed, input_builder, model)
            if res is not None:
                fout.write(json.dumps(res) + "\n")
                fout.flush()
        except Exception as e:
            print(f"Error on {data.get('id', '?')}: {e}")
            continue
finally:
    fout.close()

print("\n=== Evaluation ===")
eval_path_result_w_ans(predictions_file)

# Save to Google Drive if available
if IN_COLAB and DRIVE_BASE:
    import shutil
    drive_output = os.path.join(DRIVE_BASE, SELECTED_DATASET, model_short, SPLIT, postfix)
    os.makedirs(drive_output, exist_ok=True)
    shutil.copy2(predictions_file, os.path.join(drive_output, "predictions.jsonl"))
    print(f"Results saved to Drive: {drive_output}")

## Compare with Published Results

In [ ]:
print(
    """=== GCR Published Results (Luo et al. 2025) ===

                    WebQSP     CWQ
  GCR (Llama-3.1-8B)
    Hits@1           92.6      75.8
    F1               89.8      69.4
    Faithfulness    100%      100%

Your reproduction should be within 1-2% of these."""
)